In [0]:
# ============================================================
# NOTEBOOK 05: UNIVERSAL SILVER TRANSFORMATION EXECUTOR
# ============================================================
#
# PURPOSE:
# Apply metadata-driven transformations to a Spark DataFrame.
#
# INPUTS:
#   1. Bronze Spark DataFrame
#   2. transformation_rules prepared by Notebook 04
#
# OUTPUT:
#   Transformed Spark DataFrame
#
# SUPPORTED TRANSFORMATIONS:
#   TRIM
#   UPPERCASE
#   LOWERCASE
#   NULL_IF_EMPTY
#   RENAME
#
# DATASET INDEPENDENT:
# No dataset names or business column names are hardcoded.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from typing import Any, Dict, List

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [0]:
# ============================================================
# 2. SUPPORTED TRANSFORMATIONS
# ============================================================

SUPPORTED_TRANSFORMATIONS = {
    "TRIM",
    "UPPERCASE",
    "LOWERCASE",
    "NULL_IF_EMPTY",
    "RENAME"
}

STRING_TRANSFORMATIONS = {
    "TRIM",
    "UPPERCASE",
    "LOWERCASE",
    "NULL_IF_EMPTY"
}

In [0]:
# ============================================================
# 3. HELPER: FIND COLUMN CASE-INSENSITIVELY
# ============================================================

def find_column(
    df: DataFrame,
    requested_column: str
) -> str:

    """
    Finds the actual DataFrame column name without
    considering uppercase/lowercase differences.

    Example:
        Requested: CUSTOMER_NAME
        Actual:    customer_name
    """

    if not isinstance(requested_column, str):
        raise TypeError("Column name must be a string.")

    requested_column = requested_column.strip()

    if not requested_column:
        raise ValueError("Column name cannot be empty.")

    matches = [
        column
        for column in df.columns
        if column.lower() == requested_column.lower()
    ]

    if len(matches) == 0:
        raise ValueError(
            f"Column '{requested_column}' does not exist in DataFrame."
        )

    if len(matches) > 1:
        raise ValueError(
            f"Multiple columns match '{requested_column}' "
            f"case-insensitively: {matches}"
        )

    return matches[0]

In [0]:
# ============================================================
# 4. HELPER: SAFELY REFERENCE A COLUMN
# ============================================================

def quoted_column(column_name: str):

    """
    Creates a safely quoted Spark column reference.

    Handles column names containing spaces and special characters.
    """

    escaped_name = column_name.replace("`", "``")

    return F.col(f"`{escaped_name}`")

In [0]:
# ============================================================
# 5. HELPER: CHECK STRING DATA TYPE
# ============================================================

def is_string_column(
    df: DataFrame,
    column_name: str
) -> bool:

    """
    Checks whether a DataFrame column has Spark StringType.
    """

    actual_column = find_column(df, column_name)

    for field in df.schema.fields:
        if field.name == actual_column:
            return type(field.dataType) == StringType

    return False

In [0]:
# ============================================================
# 6. HELPER: VALIDATE TRANSFORMATION RULE STRUCTURE
# ============================================================

def validate_transformation_rules(
    transformation_rules: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:

    """
    Validates and normalizes transformation rule definitions.

    This function validates metadata structure.
    It does not modify the DataFrame.
    """

    if not isinstance(transformation_rules, list):
        raise TypeError(
            "transformation_rules must be a Python list."
        )

    normalized_rules = []

    seen_rule_keys = set()

    for index, rule in enumerate(transformation_rules):

        # ----------------------------------------------------
        # 6.1 Validate rule structure
        # ----------------------------------------------------

        if not isinstance(rule, dict):
            raise TypeError(
                f"Rule at position {index} must be a dictionary."
            )

        # ----------------------------------------------------
        # 6.2 Validate source column
        # ----------------------------------------------------

        source_column = rule.get("source_column")

        if (
            not isinstance(source_column, str)
            or not source_column.strip()
        ):
            raise ValueError(
                f"Invalid source_column in rule {index}."
            )

        source_column = source_column.strip()

        # ----------------------------------------------------
        # 6.3 Validate transformation type
        # ----------------------------------------------------

        transformation_type = rule.get("transformation_type")

        if (
            not isinstance(transformation_type, str)
            or not transformation_type.strip()
        ):
            raise ValueError(
                f"Invalid transformation_type "
                f"for '{source_column}'."
            )

        transformation_type = (
            transformation_type.strip().upper()
        )

        if transformation_type not in SUPPORTED_TRANSFORMATIONS:
            raise ValueError(
                f"Unsupported transformation "
                f"'{transformation_type}' "
                f"for '{source_column}'."
            )

        # ----------------------------------------------------
        # 6.4 Validate target column
        # ----------------------------------------------------

        target_column = rule.get("target_column")

        if (
            not isinstance(target_column, str)
            or not target_column.strip()
        ):
            raise ValueError(
                f"Invalid target_column for '{source_column}'."
            )

        target_column = target_column.strip()

        # ----------------------------------------------------
        # 6.5 Validate execution order
        # ----------------------------------------------------

        execution_order = rule.get("execution_order")

        if (
            isinstance(execution_order, bool)
            or not isinstance(execution_order, int)
            or execution_order < 1
        ):
            raise ValueError(
                f"Invalid execution_order for '{source_column}'. "
                f"Expected a positive integer."
            )

        # ----------------------------------------------------
        # 6.6 Validate transformation parameters
        # ----------------------------------------------------

        parameters = rule.get("transformation_parameters")

        if parameters is not None and not isinstance(parameters, str):
            raise ValueError(
                f"Invalid transformation_parameters for "
                f"'{source_column}'. Expected string or NULL."
            )

        # ----------------------------------------------------
        # 6.7 Validate duplicate rules
        # ----------------------------------------------------

        rule_key = (
            source_column.lower(),
            transformation_type
        )

        if rule_key in seen_rule_keys:
            raise ValueError(
                f"Duplicate transformation rule found for "
                f"'{source_column}' and '{transformation_type}'."
            )

        seen_rule_keys.add(rule_key)

        # ----------------------------------------------------
        # 6.8 Prepare normalized rule
        # ----------------------------------------------------

        normalized_rules.append({

            "original_index": index,

            "dataset_name": rule.get("dataset_name"),

            "source_column": source_column,

            "target_column": target_column,

            "transformation_type": transformation_type,

            "transformation_parameters": parameters,

            "execution_order": execution_order,

            "description": rule.get("description")

        })

    return normalized_rules

In [0]:

# ============================================================
# 7. MAIN TRANSFORMATION EXECUTOR
# ============================================================

def apply_silver_transformations(
    df: DataFrame,
    transformation_rules: List[Dict[str, Any]]
) -> DataFrame:

    """
    Applies metadata-driven transformations to a Spark DataFrame.

    Parameters:
        df:
            Input Bronze DataFrame.

        transformation_rules:
            Transformation configurations prepared by Notebook 04.

    Returns:
        Transformed Spark DataFrame.
    """

    # --------------------------------------------------------
    # 7.1 Validate input DataFrame
    # --------------------------------------------------------

    is_spark_dataframe = (
        df is not None
        and hasattr(df, "columns")
        and hasattr(df, "schema")
        and callable(getattr(df, "select", None))
        and callable(getattr(df, "withColumn", None))
        and callable(getattr(df, "withColumnRenamed", None))
    )

    if not is_spark_dataframe:
        raise TypeError(
            "Input must be a Spark DataFrame. "
            f"Received type: {type(df)}"
        )

    # --------------------------------------------------------
    # 7.2 Handle empty transformation configuration
    # --------------------------------------------------------

    if not transformation_rules:

        print(
            "No transformation rules found. "
            "Returning the original DataFrame."
        )

        return df

    # --------------------------------------------------------
    # 7.3 Validate all rule definitions
    # --------------------------------------------------------

    normalized_rules = validate_transformation_rules(
        transformation_rules
    )

    # --------------------------------------------------------
    # 7.4 Sort rules by execution order
    # --------------------------------------------------------

    sorted_rules = sorted(
        normalized_rules,
        key=lambda rule: (
            rule["execution_order"],
            rule["original_index"]
        )
    )

    print(
        f"Starting Silver transformations: "
        f"{len(sorted_rules)} rules."
    )

    # --------------------------------------------------------
    # 7.5 Initialize working DataFrame
    # --------------------------------------------------------

    result_df = df

    working_columns = {}

    # --------------------------------------------------------
    # 7.6 Execute transformation rules
    # --------------------------------------------------------

    for rule in sorted_rules:

        source_column = rule["source_column"]
        target_column = rule["target_column"]
        transformation_type = rule["transformation_type"]

        print(
            f"Applying {transformation_type} "
            f"on '{source_column}' "
            f"with target '{target_column}'."
        )

        # Resolve current source column
        source_column = source_column.lower()

        if source_column in working_columns:

            actual_column = find_column(
                result_df,
                working_columns[source_column]
            )

        else:

            actual_column = find_column(
                result_df,
                source_column
            )

        column_expression = quoted_column(actual_column)

        # ----------------------------------------------------
        # 7.7 RENAME
        # ----------------------------------------------------

        if transformation_type == "RENAME":

            if target_column == actual_column:

                print(
                    f"Source and target are identical. "
                    f"Skipping RENAME for '{actual_column}'."
                )

                working_columns[source_column] = actual_column
                continue

            if target_column.lower() == actual_column.lower():

                raise ValueError(
                    f"Case-only rename is not supported: "
                    f"'{actual_column}' to '{target_column}'."
                )

            existing_targets = [
                column
                for column in result_df.columns
                if column.lower() == target_column.lower()
            ]

            if existing_targets:

                raise ValueError(
                    f"Cannot rename '{actual_column}' to "
                    f"'{target_column}'. Target already exists."
                )

            result_df = result_df.withColumnRenamed(
                actual_column,
                target_column
            )

            working_columns[source_column] = target_column

            for key, value in list(working_columns.items()):

                if value == actual_column:
                    working_columns[key] = target_column

            continue

        # ----------------------------------------------------
        # 7.8 Validate string transformation data type
        # ----------------------------------------------------

        if transformation_type in STRING_TRANSFORMATIONS:

            if not is_string_column(
                result_df,
                actual_column
            ):

                raise TypeError(
                    f"{transformation_type} requires a string "
                    f"column. '{actual_column}' has an " 
                    f"incompatible data type."
                )

        # ----------------------------------------------------
        # 7.9 Resolve target column
        # ----------------------------------------------------

        target_matches = [
            column
            for column in result_df.columns
            if column.lower() == target_column.lower()
        ]

        if len(target_matches) > 1:

            raise ValueError(
                f"Multiple columns match target '{target_column}'."
            )

        if target_matches:

            existing_target = target_matches[0]

            if existing_target.lower() != actual_column.lower():

                raise ValueError(
                    f"Target column '{target_column}' already exists "
                    f"and is different from source '{actual_column}'."
                )

            output_column = existing_target

        else:

            output_column = target_column

        # ----------------------------------------------------
        # 7.10 TRIM
        # ----------------------------------------------------

        if transformation_type == "TRIM":

            result_df = result_df.withColumn(
                output_column,
                F.trim(column_expression)
            )

        # ----------------------------------------------------
        # 7.11 UPPERCASE
        # ----------------------------------------------------

        elif transformation_type == "UPPERCASE":

            result_df = result_df.withColumn(
                output_column,
                F.upper(column_expression)
            )

        # ----------------------------------------------------
        # 7.12 LOWERCASE
        # ----------------------------------------------------

        elif transformation_type == "LOWERCASE":

            result_df = result_df.withColumn(
                output_column,
                F.lower(column_expression)
            )

        # ----------------------------------------------------
        # 7.13 NULL_IF_EMPTY
        # ----------------------------------------------------
                                                     
        elif transformation_type == "NULL_IF_EMPTY":

            result_df = result_df.withColumn(
                output_column,
                F.when(
                    F.trim(column_expression) == "",
                    F.lit(None).cast(StringType())
                ).otherwise(
                    column_expression
                )
            )

        # Update logical source-to-output mapping
        working_columns[source_column] = output_column

    # ========================================================
    # 8. TRANSFORMATION COMPLETION
    # ========================================================

    print(
        "All configured transformations completed successfully."
    )

    return result_df



In [0]:
# ============================================================
# 9. OPTIONAL: TRANSFORMATION EXECUTOR SUMMARY
# ============================================================

def show_transformation_summary(
    transformation_rules: List[Dict[str, Any]]
):

    """
    Displays a readable summary of transformation instructions.
    Does not execute transformations.
    """

    if not transformation_rules:

        print("No transformation rules to display.")

        return

    normalized_rules = validate_transformation_rules(
        transformation_rules
    )

    sorted_rules = sorted(
        normalized_rules,
        key=lambda rule: (
            rule["execution_order"],
            rule["original_index"]
        )
    )

    print("=" * 75)
    print("SILVER TRANSFORMATION CONFIGURATION")
    print("=" * 75)

    for rule in sorted_rules:

        print(
            f"Order: {rule['execution_order']} | "
            f"Source: {rule['source_column']} | "
            f"Target: {rule['target_column']} | "
            f"Transformation: {rule['transformation_type']}"
        )

        if rule["description"]:

            print(
                f"Description: {rule['description']}"
            )

    print("=" * 75)
    print(f"Total rules: {len(sorted_rules)}")
    print("=" * 75)


# ============================================================
# NOTEBOOK 05 COMPLETE
# ============================================================
#
# This notebook defines reusable functions.
#
# It does NOT:
#   - Load the Bronze Delta table.
#   - Load transformation metadata.
#   - Run data quality checks.
#   - Write the Silver Delta table.
#
# The Silver orchestration notebook supplies the DataFrame
# and transformation rules when calling the executor.
# ============================================================
